# Appendix. 02 - LCS preprocessing and averaging

This notebook prepares the **averaged LCS trip table** used in the OD-level comparison with HTS.

The workflow:
1. loads and combines the 24 hourly LCS files for October 2021,
2. filters records to the designated Thursdays,
3. standardizes variables to the common HTS-LCS schema,
4. aligns the spatial scope to the common Seoul OD system,
5. averages LCS trips across the four designated Thursdays,
6. builds the common-format and OD-level output tables.


In [ ]:
from pathlib import Path
import itertools
import numpy as np
import pandas as pd

# Base directories
BASE_DIR = Path.cwd()
RAW_DIR = BASE_DIR / "0_data"
PREP_DIR = BASE_DIR / "1_data_preprocessing"
PREP_DIR.mkdir(parents=True, exist_ok=True)

RESULT_DIR = BASE_DIR / "2_data_OD"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

# Raw LCS files
LCS_FILE = RAW_DIR / "생활이동_행정동_202110" # https://data.seoul.go.kr
LCS_FOLDER = LCS_FILE

# Auxiliary file
ZONE_ID_FILE = RAW_DIR / "administrative districts (2016-2021).xlsx"

# Output files
LCS_TRIP_AVERAGED_FILE = PREP_DIR / "LCS_trip_averaged.csv"

OD_FRAME_FILE = RESULT_DIR / "OD_FRAME.csv"
OD_LCS_FILE = RESULT_DIR / "OD_LCS.csv"
OD_LCS_GENDER_FILE = RESULT_DIR / "OD_LCS_gender.csv"
OD_LCS_AGEGROUP_FILE = RESULT_DIR / "OD_LCS_agegroup.csv"
OD_LCS_TRAVELTYPE_FILE = RESULT_DIR / "OD_LCS_traveltype.csv"
OD_LCS_TIMEZONE_FILE = RESULT_DIR / "OD_LCS_timezone.csv"
OD_LCS_TRAVELTYPE_TIMEZONE_FILE = RESULT_DIR / "OD_LCS_traveltype_timezone.csv"

AGE_GROUPS = [
    "0-9 years",
    "10-19 years",
    "20-29 years",
    "30-39 years",
    "40-49 years",
    "50-59 years",
    "60-69 years",
    "70-79 years",
    "80 years and above",
]

GENDER_MAP = {"M": "Male", "F": "Female"}

# Number of Thursdays in October 2021
N_DESIGNATED_THURSDAYS = 4

In [ ]:
def age_to_group(age: pd.Series) -> pd.Categorical:
    bins = [0, 10, 20, 30, 40, 50, 60, 70, 80, np.inf]
    grouped = pd.cut(age, bins=bins, labels=AGE_GROUPS, right=False, include_lowest=True)
    return pd.Categorical(grouped, categories=AGE_GROUPS, ordered=True)

def assign_time_zone(hour: pd.Series) -> pd.Series:
    hour = pd.to_numeric(hour, errors="coerce").mod(24)
    out = pd.Series(np.nan, index=hour.index, dtype="object")
    out.loc[hour.between(7, 10)] = "T1"
    out.loc[hour.between(11, 14)] = "T2"
    out.loc[hour.between(15, 18)] = "T3"
    out.loc[hour.between(19, 22)] = "T4"
    out.loc[(hour == 23) | hour.between(0, 2)] = "T5"
    out.loc[hour.between(3, 6)] = "T6"
    return out

def standardize_zone_id(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce").astype("Int64").astype("string")

def standardize_gender(series: pd.Series) -> pd.Series:
    mapped = series.replace(GENDER_MAP)
    mapped = mapped.astype("string").str.strip()
    return mapped.replace(GENDER_MAP)

def build_seoul_code_lookup(zone_id_file: Path) -> list[str]:
    lookup = pd.read_excel(zone_id_file)
    lookup = lookup.loc[lookup["시도"].eq("서울특별시")].copy()

    zone_ids = (
        standardize_zone_id(lookup["2016_행정구역분류(통계청)"]) # "zone_id"
        .dropna()
        .drop_duplicates()
        .sort_values()
        .tolist()
    )
    return zone_ids


## 1. Load and combine raw LCS records

In [ ]:
from tqdm.auto import tqdm

def load_lcs_hourly_files(lcs_folder: Path, show_progress: bool = True) -> pd.DataFrame:
    # Read all 24 hourly LCS files for October 2021 and combine them
    # into one raw table before attribute harmonization.
    file_list = [
        lcs_folder / f"생활이동_행정동_2021.10_{hour:02d}시.csv"
        for hour in range(24)
    ]

    # Stop if any hourly source file is missing.
    missing_files = [str(file) for file in file_list if not file.exists()]
    if missing_files:
        raise FileNotFoundError(
            "The following hourly LCS files are missing:\n" + "\n".join(missing_files)
        )

    # Read each hourly file with a visible progress bar.
    iterator = tqdm(file_list, desc="Loading LCS hourly files") if show_progress else file_list
    lcs_parts = []

    for file in iterator:
        if not show_progress:
            print(f"Reading: {file.name}")
        lcs_parts.append(pd.read_csv(file, encoding="cp949"))

    # Stack all hourly records into a single raw DataFrame.
    return pd.concat(lcs_parts, ignore_index=True)

lcs_raw = load_lcs_hourly_files(LCS_FOLDER)

print(f"Raw LCS rows: {len(lcs_raw):,}")
lcs_raw

## 2. Load and preprocess LCS trip records

In [ ]:
def preprocess_lcs_trip(lcs_raw: pd.DataFrame) -> pd.DataFrame:
    df = lcs_raw.copy()

    # Rename raw LTE/5G Cellular Signaling data variables to descriptive column names.
    df = df.rename(columns={
        "요일": "day_of_week",
        "출발 행정동 코드": "origin_id",
        "도착 행정동 코드": "destination_id",
        "성별": "gender",
        "나이": "age",
        "이동유형": "travel_type",
        "도착시간": "arrival_hour",
        "평균 이동 시간(분)": "travel_time_min",
        "이동인구(합)": "trips",
    })

    # Restrict the LCS data to Thursday trips to match the HTS survey day.
    # LCS zone IDs already match the 2016 analysis-zone codes, so no additional crosswalk is required.
    df = df.loc[df["day_of_week"].eq("목")].copy() 

    # Standardize data types and clean string fields.
    # Age is also converted into the common analysis age groups.
    df["origin_id"] = standardize_zone_id(df["origin_id"])
    df["destination_id"] = standardize_zone_id(df["destination_id"])
    df["gender"] = standardize_gender(df["gender"])
    df["age"] = pd.to_numeric(df["age"], errors="coerce")
    df["age_group"] = age_to_group(df["age"])

    # Harmonize travel-type codes to the common H/W/O-based format
    # and exclude within-home trips from the analysis.
    df["travel_type"] = (
        df["travel_type"]
        .astype("string")
        .str.strip()
        .str.upper()
        .str.replace("E", "O", regex=False) # "E" = "O"= "Other
    )

    df = df.loc[df["travel_type"].ne("HH")].copy()

    # Use arrival time as the common temporal reference
    # and assign each trip to one of the six time zones.
    df["arrival_hour"] = pd.to_numeric(df["arrival_hour"], errors="coerce")
    df["time_zone"] = assign_time_zone(df["arrival_hour"])

    # Ensure numeric fields are stored in numeric format.
    df["travel_time_min"] = pd.to_numeric(df["travel_time_min"], errors="coerce")
    df["trips"] = pd.to_numeric(df["trips"], errors="coerce")

    # Drop rows where trips is "*", NaN, or 0.
    before_drop = len(df)
    df = df.loc[df["trips"].ne("*")].copy()
    df["trips"] = pd.to_numeric(df["trips"], errors="coerce")
    df = df.loc[df["trips"].notna() & df["trips"].ne(0)].copy()
    after_drop = len(df)
    print(f"Removed {before_drop - after_drop:,} rows with invalid trips values (*, NaN, or 0).")

    keep_cols = [
        "origin_id",
        "destination_id",
        "gender",
        "age_group",
        "travel_type",
        "arrival_hour",
        "time_zone",
        "travel_time_min",
        "trips",
    ]
    return df[keep_cols].copy()

lcs_trip_raw = preprocess_lcs_trip(lcs_raw)
lcs_trip_raw

In [ ]:
print(f"Raw LCS rows: {len(lcs_raw):,}")
print(f"Thursday LCS rows before Seoul OD filtering: {len(lcs_trip_raw):,}")

print(f"Thursday LCS trips before Seoul OD filtering (4 days): {lcs_trip_raw['trips'].sum():,.0f}")

## 3. Align spatial scope to the common Seoul OD system

In [ ]:
def filter_lcs_to_seoul_od_scope(
    lcs_trip_raw: pd.DataFrame,
    zone_id_file: Path,
) -> pd.DataFrame:
    # Load the set of Seoul administrative-zone IDs
    # used in the common HTS-LCS OD framework.
    zone_ids = set(build_seoul_code_lookup(zone_id_file))

    # Retain only trips whose origin and destination
    # both fall within the Seoul analysis zone system.
    # Also remove records with missing trip volumes.
    df = lcs_trip_raw.loc[
        lcs_trip_raw["origin_id"].isin(zone_ids)
        & lcs_trip_raw["destination_id"].isin(zone_ids)
        & lcs_trip_raw["trips"].notna()
    ].copy()

    # Reset the row index after filtering.
    return df.reset_index(drop=True)

LCS_TRIP = filter_lcs_to_seoul_od_scope(
    lcs_trip_raw=lcs_trip_raw,
    zone_id_file=ZONE_ID_FILE,
)

print(f"Seoul LCS rows after OD filtering: {len(LCS_TRIP):,}")
print(f"Thursday LCS trips after Seoul OD filtering (4 days): {LCS_TRIP['trips'].sum():,.0f}")

print(f"Number of Seoul origin zones: {LCS_TRIP['origin_id'].nunique():,}")
print(f"Number of Seoul destination zones: {LCS_TRIP['destination_id'].nunique():,}")

LCS_TRIP

## 4. Average LCS trips across designated Thursdays

In [ ]:
def average_lcs_trips(
    lcs_trip: pd.DataFrame,
    n_designated_days: int = N_DESIGNATED_THURSDAYS,
) -> pd.DataFrame:
    # Convert the summed LCS trip counts across designated Thursdays
    # into the average trip volume for one designated Thursday.
    df = lcs_trip.copy()
    df["trips"] = df["trips"] / n_designated_days
    return df

LCS_TRIP_AVERAGED = average_lcs_trips(LCS_TRIP)

# SAVE
LCS_TRIP_AVERAGED.to_csv(LCS_TRIP_AVERAGED_FILE, index=False, encoding="utf-8-sig")

print(f"Seoul LCS rows after OD filtering: {len(LCS_TRIP_AVERAGED):,}")
print(f"Thursday LCS trips after Seoul OD filtering (1 days): {LCS_TRIP_AVERAGED['trips'].sum():,.0f}")

print('Save complete')
# print(f"Saved: {LCS_TRIP_AVERAGED_FILE}")

LCS_TRIP_AVERAGED

## 5. Convert averaged LCS trips to the common HTS-LCS format

In [ ]:
def prepare_lcs_trip_keys(lcs_trip_averaged: pd.DataFrame) -> pd.DataFrame:
    """
    Standardize OD key columns in the averaged LCS trip table.
    """
    df = lcs_trip_averaged.copy()

    df["origin_id"] = standardize_zone_id(df["origin_id"])
    df["destination_id"] = standardize_zone_id(df["destination_id"])

    return df


def build_od_frame(zone_id_file: Path) -> pd.DataFrame:
    """
    Build the complete Seoul OD frame (424 × 424) from the zone-id file.
    The OD keys are stored as strings for merge stability.
    """
    zone_lookup = pd.read_excel(zone_id_file)

    zone_lookup = zone_lookup.loc[
        zone_lookup["시도"].eq("서울특별시"),
        ["2016_행정구역분류(통계청)"],
    ].copy()

    zone_ids = (
        pd.to_numeric(zone_lookup["2016_행정구역분류(통계청)"], errors="coerce")
        .dropna()
        .astype(np.int64)
        .astype(str)
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    od_pairs = list(itertools.product(zone_ids, repeat=2))
    od_df = pd.DataFrame(od_pairs, columns=["origin_id", "destination_id"])

    od_df["OD_pair"] = od_df["origin_id"] + "_" + od_df["destination_id"]

    return od_df[["OD_pair", "origin_id", "destination_id"]].copy()


def aggregate_od_trips(
    lcs_trip_averaged: pd.DataFrame,
    od_frame: pd.DataFrame,
) -> pd.DataFrame:
    """
    Aggregate averaged LCS trips to the OD level only.
    Unobserved OD cells are filled with zero.
    """
    df = prepare_lcs_trip_keys(lcs_trip_averaged)

    observed = (
        df.groupby(["origin_id", "destination_id"], dropna=False, as_index=False)["trips"]
        .sum()
    )

    table = od_frame.merge(
        observed,
        on=["origin_id", "destination_id"],
        how="left",
    )

    table["trips"] = table["trips"].fillna(0)

    table = table[
        ["OD_pair", "origin_id", "destination_id", "trips"]
    ].sort_values(["origin_id", "destination_id"]).reset_index(drop=True)

    return table


def build_attribute_levels(
    lcs_trip_averaged: pd.DataFrame,
    attr_cols: list[str],
) -> pd.DataFrame:
    """
    Extract unique attribute combinations observed in the averaged LCS trip table.
    """
    df = lcs_trip_averaged.copy()

    levels = (
        df[attr_cols]
        .drop_duplicates()
        .sort_values(attr_cols)
        .reset_index(drop=True)
    )

    return levels


def build_od_attribute_table(
    lcs_trip_averaged: pd.DataFrame,
    od_frame: pd.DataFrame,
    attr_col: str,
) -> pd.DataFrame:
    """
    Build an OD × single-attribute averaged trip table.
    The table contains all OD pairs crossed with all observed levels
    of the given attribute, with missing cells filled with zero.
    """
    df = prepare_lcs_trip_keys(lcs_trip_averaged)

    observed = (
        df.groupby(
            ["origin_id", "destination_id", attr_col],
            dropna=False,
            as_index=False,
        )["trips"]
        .sum()
    )

    attr_levels = build_attribute_levels(df, [attr_col])

    # Cross join: OD × attribute levels
    full_frame = od_frame.merge(attr_levels, how="cross")

    table = full_frame.merge(
        observed,
        on=["origin_id", "destination_id", attr_col],
        how="left",
    )

    table["trips"] = table["trips"].fillna(0)

    table = table[
        ["OD_pair", "origin_id", "destination_id", attr_col, "trips"]
    ].sort_values(["origin_id", "destination_id", attr_col]).reset_index(drop=True)

    return table


def build_od_multi_attribute_table(
    lcs_trip_averaged: pd.DataFrame,
    od_frame: pd.DataFrame,
    attr_cols: list[str],
) -> pd.DataFrame:
    """
    Build an OD × multi-attribute averaged trip table.
    The table contains all OD pairs crossed with all observed attribute
    combinations, with missing cells filled with zero.
    """
    df = prepare_lcs_trip_keys(lcs_trip_averaged)

    observed = (
        df.groupby(
            ["origin_id", "destination_id"] + attr_cols,
            dropna=False,
            as_index=False,
        )["trips"]
        .sum()
    )

    attr_levels = build_attribute_levels(df, attr_cols)

    # Cross join: OD × attribute-combination levels
    full_frame = od_frame.merge(attr_levels, how="cross")

    table = full_frame.merge(
        observed,
        on=["origin_id", "destination_id"] + attr_cols,
        how="left",
    )

    table["trips"] = table["trips"].fillna(0)

    table = table[
        ["OD_pair", "origin_id", "destination_id"] + attr_cols + ["trips"]
    ].sort_values(["origin_id", "destination_id"] + attr_cols).reset_index(drop=True)

    return table

In [ ]:
OD_FRAME = build_od_frame(ZONE_ID_FILE)

print(f"Number of Seoul zones: {OD_FRAME['origin_id'].nunique():,}")
print(f"Number of OD pairs: {len(OD_FRAME):,}")

OD_LCS = aggregate_od_trips(
    lcs_trip_averaged=LCS_TRIP_AVERAGED,
    od_frame=OD_FRAME,
)

OD_LCS_gender = build_od_attribute_table(
    lcs_trip_averaged=LCS_TRIP_AVERAGED,
    od_frame=OD_FRAME,
    attr_col="gender",
)

OD_LCS_agegroup = build_od_attribute_table(
    lcs_trip_averaged=LCS_TRIP_AVERAGED,
    od_frame=OD_FRAME,
    attr_col="age_group",
)

OD_LCS_traveltype = build_od_attribute_table(
    lcs_trip_averaged=LCS_TRIP_AVERAGED,
    od_frame=OD_FRAME,
    attr_col="travel_type",
)

OD_LCS_timezone = build_od_attribute_table(
    lcs_trip_averaged=LCS_TRIP_AVERAGED,
    od_frame=OD_FRAME,
    attr_col="time_zone",
)

OD_LCS_traveltype_timezone = build_od_multi_attribute_table(
    lcs_trip_averaged=LCS_TRIP_AVERAGED,
    od_frame=OD_FRAME,
    attr_cols=["travel_type", "time_zone"],
)

print("OD matrix creation by subgroup completed")

In [ ]:
OD_LCS_gender

In [ ]:
OD_LCS_agegroup

In [ ]:
OD_LCS_traveltype

In [ ]:
OD_LCS_timezone

In [ ]:
OD_LCS_traveltype_timezone

In [ ]:
# Save
OD_LCS.to_csv(OD_LCS_FILE, index=False, encoding="utf-8-sig")
OD_LCS_gender.to_csv(OD_LCS_GENDER_FILE, index=False, encoding="utf-8-sig")
OD_LCS_agegroup.to_csv(OD_LCS_AGEGROUP_FILE, index=False, encoding="utf-8-sig")
OD_LCS_traveltype.to_csv(OD_LCS_TRAVELTYPE_FILE, index=False, encoding="utf-8-sig")
OD_LCS_timezone.to_csv(OD_LCS_TIMEZONE_FILE, index=False, encoding="utf-8-sig")
OD_LCS_traveltype_timezone.to_csv(OD_LCS_TRAVELTYPE_TIMEZONE_FILE, index=False, encoding="utf-8-sig")

print('Save complete')
# print(f"Saved: {OD_FRAME_FILE}")
# print(f"Saved: {OD_LCS_FILE}")
# print(f"Saved: {OD_LCS_GENDER_FILE}")
# print(f"Saved: {OD_LCS_AGEGROUP_FILE}")
# print(f"Saved: {OD_LCS_TRAVELTYPE_FILE}")
# print(f"Saved: {OD_LCS_TIMEZONE_FILE}")
# print(f"Saved: {OD_LCS_TRAVELTYPE_TIMEZONE_FILE}")


### (Additional Analysis) Subgroup-Level Trip Distribution (Percent and Count)

In [ ]:
# Summarize raw and LCS trip distributions
# by gender, age group, travel type, and time zone.

def summarize_trip_distribution(
    df: pd.DataFrame,
    group_col: str,
) -> pd.DataFrame:
    total_trips = df["trips"].sum()

    summary = (
        df.groupby(group_col, dropna=False, as_index=False)
        .agg(trips=("trips", "sum"))
    )

    summary["trip_pct"] = summary["trips"] / total_trips * 100
    return summary.sort_values(group_col).reset_index(drop=True)

def format_distribution_table(df: pd.DataFrame, group_col: str) -> pd.DataFrame:
    out = df.copy()
    out["trips"] = out["trips"].round(0).astype(int)
    out["trip_pct"] = out["trip_pct"].round(1)
    return out.rename(columns={
        group_col: "category",
        "trips": "Average LCS trips (n)",
        "trip_pct": "Average LCS trips (%)",
    })

LCS_DIST_GENDER = format_distribution_table(
    summarize_trip_distribution(LCS_TRIP_AVERAGED, "gender"),
    "gender",
)

LCS_DIST_AGEGROUP = format_distribution_table(
    summarize_trip_distribution(LCS_TRIP_AVERAGED, "age_group"),
    "age_group",
)

LCS_DIST_TRAVELTYPE = format_distribution_table(
    summarize_trip_distribution(LCS_TRIP_AVERAGED, "travel_type"),
    "travel_type",
)

LCS_DIST_TIMEZONE = format_distribution_table(
    summarize_trip_distribution(LCS_TRIP_AVERAGED, "time_zone"),
    "time_zone",
)

print("\n[Gender distribution]")
display(LCS_DIST_GENDER)

print("\n[Age-group distribution]")
display(LCS_DIST_AGEGROUP)

print("\n[Travel-type distribution]")
display(LCS_DIST_TRAVELTYPE)

print("\n[Time-zone distribution]")
display(LCS_DIST_TIMEZONE)


## Notes

- The LCS files are averaged across the **four designated Thursdays** in October 2021.
- `travel_type = "HH"` is removed to align the LCS scope with the HTS comparison framework.
- The final common-format columns are matched to the HTS notebook:
  `origin_id`, `destination_id`, `gender`, `age_group`, `travel_type`,
  `time_zone`, `arrival_hour`, `travel_time_min`, and `trips`.
